In [1]:
import pandas as pd, numpy as np, ee
ee.Initialize(project='sentinel-500611')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [60]:
"""
Utwórz DataFrame zawierający dane 10 osób, wzrost w centymetrach oraz
masę ciała w kilogramach. Następnie na podstawie tych
danych wygeneruj kolumnę zawierającą BMI każdej osoby.
"""

wzrost = np.random.randint(156, 193, (10, 1))
waga = np.random.randint(50, 90, (10, 1))
arr = np.concatenate([wzrost, waga], axis=1)

df = pd.DataFrame(arr, columns=['wzrost', 'waga'])
df['imie'] = ['Adam', 'Małgorzata', 'Agata', 'Piotr', 'Barbara', 'Joanna', 'Arkadiusz', 'Hanna', 'Jan', 'Filip']
df = df[['imie', 'wzrost', 'waga']]
df['bmi'] = df['waga'] / ((df['wzrost'] / 100 ) ** 2 )
print(df)
df_bmi = df

         imie  wzrost  waga        bmi
0        Adam     158    52  20.829995
1  Małgorzata     160    58  22.656250
2       Agata     159    87  34.413196
3       Piotr     170    79  27.335640
4     Barbara     162    68  25.910684
5      Joanna     190    54  14.958449
6   Arkadiusz     167    78  27.968016
7       Hanna     182    63  19.019442
8         Jan     186    84  24.280264
9       Filip     172    68  22.985398


In [36]:
"""
zadanie 2
Utwórz własny DataFrame zawierający wybrane przez siebie dane. Zapisz go do pliku JSON, a następnie wczytaj ten plik do nowego obiektu DataFrame. Sprawdź, czy oba obiekty zawierają te same dane.
"""

from pathlib import Path

collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").first()
bands_info = collection.getInfo()['bands']
bands_df = pd.DataFrame(
    {
        'name': [name['id'] for name in bands_info],
        'resolution_m': [res['crs_transform'][0] for res in bands_info],
        'crs': [c['crs'] for c in bands_info],
    }
)

current_path = Path.cwd()
json_path = str(current_path / 'bands_df.json')
bands_df.to_json(json_path)

In [74]:
bands_df_json = pd.read_json(json_path)
print(bands_df_json.equals(bands_df))



True


In [79]:
"""
zadanie 3
Wykorzystaj DataFrame stworzony w punkcie 1. Dodaj kolumnę z wiekiem i wypełnij ją danymi. Następnie znajdź osoby starsze niż średnia wieku wszystkich osób, których BMI jest jednocześnie większe niż 25. Wyświetl imiona, wiek oraz BMI tych osób.
"""

from random import randint

age = [randint(18, 70) for i in range(10)]
df_bmi['wiek'] = age

mean_age = df_bmi['wiek'].mean()
result = df.loc[(df['wiek'] > mean_age) & (df['bmi'] > 25), ['imie', 'wiek', 'bmi']]
print(f"średni wiek: {mean_age}", result, sep='\n')




średni wiek: 45.4
    imie  wiek        bmi
2  Agata    63  34.413196
3  Piotr    62  27.335640


In [1]:
def select_top_extremes(df, criteria):
    n, period = criteria

    # Kopia DataFrame i konwersja daty
    df_temp = df.copy()

    # Wybór okresu grupowania ('year' lub 'month')
    if period == 'year':
        df_temp['period'] = df_temp['date'].dt.to_period('Y')
    elif period == 'month':
        df_temp['period'] = df_temp['date'].dt.to_period('M')
    else:
        raise ValueError("Drugi element krotki musi to być 'year' lub 'month'")

    # Priorytet: najpierw sortuj po x, a przy remisie po y
    sorted_df = df_temp.sort_values(by=['period', 'clouds', 'snow'])

    # Pobranie n pierwszych wierszy dla każdego okresu i czyszczenie pomocniczych kolumn
    result = sorted_df.groupby('period').head(n)
    return result.drop(columns=['period', 'score'])

NameError: name 'df' is not defined